In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.stores import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers.document_compressors import cohere_rerank
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder
from langsmith import traceable
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, field_validator
from dotenv import load_dotenv
from torch import embedding
# from all_document.data import documents
import os
from langchain_classic.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
from langchain_chroma import Chroma

g:\langc-in-production\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import textwrap
def wrap_text(text, width=90):
    #split the input text into line based on newline characters
    line = text.split('\n')

    #wrap each line individual
    wrapped_line = [textwrap.fill(l, width) for l in line]

    #join wrapper line back together using newline characters
    wrapped_text = '\n'.join(wrapped_line)

    return wrapped_text

In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model="llama-3.3-70b-versatile"
encode_kwargs = {'normalize_embeddings' : True}
embedding_model = HuggingFaceEmbeddings(model_name=model,
                                        encode_kwargs=encode_kwargs)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3911.11it/s]


In [6]:
# data_path = "G:\\langc-in-production\\all_document\\ai_agent.pdf"

# loader = DirectoryLoader(path=data_path,
#                          show_progress=True,
#                          loader_cls=PyMuPDFLoader, 
#                          glob="**/*.pdf")
# docs = loader.load()

# print(len(docs))

In [7]:
from os import path


loader = PyMuPDFLoader(data_path)

docss = loader.load()

In [8]:
print(len(docss))

288


In [10]:
print(docss[30])

page_content='Fundamentals of Generative AI
6
game elements such as character design, level layouts, music and sound effects, and so on. By 
providing different conditions such as “Create a forest level” or “Create a desert level,” the CVAE 
can produce a wide variety of game environments, saving time for designers and enhancing 
the player’s experience with more diverse and interesting game worlds [3].
GANs
A GAN is basically formed by two neural networks: a generator and a discriminator. The generator 
generates synthetic data samples; the other trained neural network should then be able to tell the 
difference between real and created samples. While training these networks, they are trained together 
antagonistically: the generator tries to fool the discriminator, while the discriminator tries rightly to 
classify real versus fake data. In this competition, the generator gets better and better at faking data. 
The following are some of the different types of GANs:
•	 GAN: The basic 

In [11]:
print(docss[90].page_content)

Essential Components of Intelligent Agents
66
•	 Optimal path finding: This is a specific type of graph search that aims to find not just any path, 
but the best path according to some criteria (usually minimizing total edge cost). Two of the 
algorithms in this category are the Bellman-Ford algorithm and the A* search.
The downsides of using graph-based planning algorithms include fixing the state representation (state 
space) upfront, and the potential for exponential growth in the number of states to represent and 
store as problems get more complex.
Graph-based planning techniques find numerous real-world applications across domains, where 
finding optimal sequences of actions to achieve goals is crucial. These applications include navigation 
and route planning, such as GPS systems using graph representations of road networks to find optimal 
routes minimizing travel time or distance. Logistics and supply chain applications involve planning 
optimal sequences of operations for man

In [12]:
doc = docss
raw_data = ''

for i, doc in enumerate(doc):
    text = doc.page_content
    if text:
        raw_data += text

In [13]:
raw_data

'Building Agentic AI Systems\nCreate intelligent, autonomous AI agents that can reason, plan, \nand adapt\nAnjanava Biswas\nWrick TalukdarBuilding Agentic AI Systems\nCopyright © 2025 Packt Publishing\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval system, or transmitted \nin any form or by any means, without the prior written permission of the publisher, except in the case \nof brief quotations embedded in critical articles or reviews.\nThe author acknowledges the use of cutting-edge AI, such as ChatGPT, with the sole aim of enhancing \nthe language and clarity within the book, thereby ensuring a smooth reading experience for readers. \nIt’s important to note that the content itself has been crafted by the author and edited by a professional \npublishing team.\nEvery effort has been made in the preparation of this book to ensure the accuracy of the information \npresented. However, the information contained in this book is sold without warranty, eit

In [14]:
len(raw_data)

605996

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len,
    is_separator_regex= False
)

In [16]:
texts = text_splitter.split_text(raw_data)

In [17]:
len(texts)

1525

In [18]:
print(texts[100])

to think about what creativity is, who the artist really is, and what the ethical parameters should be 
for AI-created content.
Having understood what generative AI is and its brief history, let’s explore the different types of 
generative AI models.
Types of generative AI models
Generative AI is an exciting domain of AI that deals with the generation of new, synthetic data by 
learning patterns from existing datasets, aiming to generate outputs that share similar statistical


In [19]:
db = Chroma.from_texts(texts=texts, embedding=embedding_model, persist_directory="./chroma_db")

In [20]:
query = "what is agentic ai"

data = db.similarity_search(query=query, k =5)

In [21]:
for i, q in enumerate(data, start=1):
    print(f"Query_Ranked:{i} = {q.page_content}\n")

Query_Ranked:1 = You can find the code file for this chapter on GitHub at https://github.com/PacktPublishing/
Building-Agentic-AI-Systems. In this chapter, we will also use agentic Python frameworks 
such as CrewAI, AutoGen, and LangChain to demonstrate the various aspects of AI agents.
Understanding the concept of tool use in agents
At its core, tool usage by an intelligent agent refers to the LLM agent’s capability of leveraging external

Query_Ranked:2 = on experience
Start building agentic AI
We have learned quite a lot about the characteristics of intelligent agents, how they are built, how they 
work with different algorithms, and their essential components. It is now time for a gentle introduction 
to the world of agentic AI and to start building applications using different frameworks.
In subsequent chapters of this book, we will make extensive use of several open source frameworks. The

Query_Ranked:3 = on experience
Start building agentic AI
We have learned quite a lot about 

### Retrieval Setup

In [22]:
retrievers = db.as_retriever(k=3)

retrievers.invoke(query)

[Document(id='a83f2b56-33b7-42cd-bf2c-079b6db42b0b', metadata={}, page_content='You can find the code file for this chapter on GitHub at https://github.com/PacktPublishing/\nBuilding-Agentic-AI-Systems. In this chapter, we will also use agentic Python frameworks \nsuch as CrewAI, AutoGen, and LangChain to demonstrate the various aspects of AI agents.\nUnderstanding the concept of tool use in agents\nAt its core, tool usage by an intelligent agent refers to the LLM agent’s capability of leveraging external'),
 Document(id='a8a3f417-4499-4282-a288-3f0092ff77e1', metadata={}, page_content='on experience\nStart building agentic AI\nWe have learned quite a lot about the characteristics of intelligent agents, how they are built, how they \nwork with different algorithms, and their essential components. It is now time for a gentle introduction \nto the world of agentic AI and to start building applications using different frameworks.\nIn subsequent chapters of this book, we will make extensiv

### Chat Chain

In [23]:
template = """Answer the following question based only from the context:
{context}

Question: {question}
"""

In [24]:
prompt = ChatPromptTemplate.from_template(template)

In [25]:
llm = init_chat_model("google_genai:gemini-2.5-flash")

In [26]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the following question based only from the context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [27]:
llm

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000167AA2C8C20>, default_metadata=(), model_kwargs={})

In [28]:
chain = (
    {"context" : retrievers, "question" : RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [47]:
text_reply = chain.invoke("how to master ai agentic")

print(wrap_text(text_reply))

Based on the context provided, to start learning and building in the world of agentic AI:

1.  Begin with a gentle introduction to agentic AI.
2.  Start building applications using different frameworks.
3.  The book will make extensive use of several open source frameworks in subsequent
chapters for this purpose.


In [36]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import ChatMessagePromptTemplate, PromptTemplate

In [37]:
from langchain_core import messages


prompt = ChatPromptTemplate(input_variables=['original_query'],
                            messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='you are a helpful assistant that generates multiples search query on a single input query.')),
                                     HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['original_query'], template='generate muliple search queries related to : {question} \n Output (4 queries):'))])


In [38]:
original_query = "how to master ai agent"

In [39]:
generates_query = (
    prompt | llm | StrOutputParser() | (lambda x:x.split("\n"))
)

In [40]:
generates_query

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='you are a helpful assistant that generates multiples search query on a single input query.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='generate muliple search queries related to : {question} \n Output (4 queries):'), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_

In [41]:
from langchain_core.load import dumps, loads

def reciprocal_fusion(results : list[list], k=60):
    fused_ranked = {}
    for docs in results:
        #assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_ranked:
                fused_ranked[doc_str] = 0
            previous_score = fused_ranked[doc_str]
            fused_ranked[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_ranked.items(), key=lambda x: x[1], reverse=True)
        ]
    return reranked_results

In [42]:
ragfusion_chain = generates_query | retrievers.map() | reciprocal_fusion

In [43]:
import langchain
langchain.debug = True

In [46]:
ragfusion_chain.input_schema.model_json_schema()

{'properties': {'question': {'title': 'Question', 'type': 'string'}},
 'required': ['question'],
 'title': 'PromptInput',
 'type': 'object'}

In [48]:
ragfusion_chain.invoke({"question":original_query})

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6.322980206s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi